# PRADEX — Narração com voz clonada (XTTS-v2) — Colab GRÁTIS

Gera a **narração em pt-BR na voz clonada do Lucas** pro vídeo faceless (Remotion).
Open-source (Coqui XTTS-v2), roda no GPU grátis do Colab. **R$0, sem API paga.**

**Ordem (briefing §3.2 — GATE primeiro):**
1. Instalar + carregar o modelo.
2. Definir o **sanitizador** (limpa o texto antes do TTS).
3. Subir o seu **sample de voz** (~1-2 min, limpo).
4. **GATE:** gerar 1 frase-teste → ouvir. **Só seguir se aprovar o clone.**
5. (Só após aprovar) gerar a narração **por cena** + `durations.json` e baixar o zip.

> Dica: Runtime → Change runtime type → **GPU (T4)** antes de rodar.

## 1. Instalar Coqui XTTS-v2

In [ ]:
# Fork mantido da Coqui (mesmo import `TTS`). Se falhar, troque por: !pip install -q TTS
!pip install -q coqui-tts
import os
os.environ['COQUI_TOS_AGREED'] = '1'  # aceita o ToS pro auto-download do XTTS-v2
print('ok')

In [ ]:
import torch
from TTS.api import TTS
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('modelo carregado')

## 2. Sanitizador de texto (anti-ruído)
Limpa o texto **antes** de mandar pro TTS: travessões viram pausa (vírgula), aspas e
símbolos soltos somem. Assim, mesmo que um roteiro futuro escape um `—`, nunca vira ruído.

In [ ]:
def sanitize(text):
    """Limpa o texto antes do TTS pra nunca virar ruído."""
    if not text:
        return ''
    for d in ('—', '–', '―'):
        text = text.replace(d, ', ')   # travessão / en-dash -> pausa natural
    for q in ('"', "'", '“', '”', '‘', '’', '«', '»', '„'):
        text = text.replace(q, '')      # aspas de qualquer tipo
    for s in ('*', '_', '#', '~', '^', '|', '/', '<', '>', '=', '+', '`', '@', '&', '[', ']', '{', '}'):
        text = text.replace(s, ' ')     # símbolos soltos
    while '  ' in text:
        text = text.replace('  ', ' ')  # espaços duplicados
    for p in (',', '.', '!', '?', ';', ':'):
        text = text.replace(' ' + p, p) # espaço antes de pontuação
    text = text.replace(',,', ',')
    return text.strip()

# teste rápido:
print(sanitize('O problema — "teste" — é não enxergar * # / e tal'))

## 3. Subir o sample de voz
WAV ou MP3, ~1-2 min, ambiente quieto, fala natural (tom de conversa).

In [ ]:
from google.colab import files
print('Suba o sample de voz (sample.wav / .mp3):')
up = files.upload()
SAMPLE = list(up.keys())[0]
print('sample:', SAMPLE)

## 4. 🚧 GATE — frase-teste do clone
Gera **uma** frase. Ouça abaixo. **Soa como você?**
- ✅ Sim → siga pra etapa 5.
- ❌ Não → pare. Sample mais limpo/longo; se ainda decepcionar, fallback (voz TTS neutra grátis) — e só em último caso, com OK explícito, uma opção paga.

In [ ]:
from IPython.display import Audio, display
TESTE = 'Seu dinheiro não some. Ele vaza. E você nem vê por onde.'
tts.tts_to_file(text=sanitize(TESTE), speaker_wav=SAMPLE, language='pt', file_path='test_clone.wav')
display(Audio('test_clone.wav'))

## 5. (Só após aprovar) Narração por cena + durações
Suba o `engine/remotion/src/script.json` (tem o campo `narracao` por cena).

In [ ]:
import json
print('Suba o script.json:')
s = files.upload()
script = json.load(open(list(s.keys())[0], encoding='utf-8'))
print(len(script['cenas']), 'cenas')

In [ ]:
import os, wave, contextlib
os.makedirs('out', exist_ok=True)
durs = {}
for c in script['cenas']:
    txt = sanitize(c.get('narracao') or '')
    if not txt:
        continue
    path = f"out/{c['id']}.wav"
    tts.tts_to_file(text=txt, speaker_wav=SAMPLE, language='pt', file_path=path)
    with contextlib.closing(wave.open(path, 'r')) as w:
        durs[c['id']] = round(w.getnframes() / float(w.getframerate()), 3)
    print(c['id'], durs[c['id']], 's')
json.dump(durs, open('out/durations.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print('total narração:', round(sum(durs.values()), 1), 's')

In [ ]:
# Baixa narracao.zip → descompacte em engine/remotion/public/narracao/
!cd out && zip -q -r ../narracao.zip .
files.download('narracao.zip')

## 6. Depois (na máquina do Lucas)
1. Descompacte `narracao.zip` em `engine/remotion/public/narracao/` (gera `*.wav` + `durations.json`).
2. Avise o Claude Code: ele re-renderiza o MP4 **com som** (só narração; a música você põe no app).